# Phase 2: IQA Model Training - Google Colab

**Project**: Image Preprocessing Detector  
**Phase**: 2 - ML for Image Quality Assessment  
**Model**: MobileNetV3-Small / EfficientNet-B0  
**Task**: Multi-label classification (6 quality issues)  

## Overview

This notebook trains a multi-label CNN classifier to detect image quality issues:
- Noise
- Blur
- Skew/Rotation
- Perspective Distortion
- Low Contrast
- Image Orientation

## Colab Pro Requirements

- **Session Limit**: 12 hours
- **GPU**: V100 (16GB), P100 (16GB), or T4 (15GB)
- **Expected Training Time**: 12-20 hours (may require 2 sessions)
- **Google Drive Space**: ~25GB (dataset + checkpoints)

## Checkpoint Management

This notebook automatically:
- Saves checkpoints every 5 epochs OR 30 minutes
- Resumes from last checkpoint if session interrupted
- Syncs checkpoints to Google Drive continuously
- Stops training at 11.5 hours (before 12hr limit)

## Instructions

1. **Runtime** → Change runtime type → GPU (V100 recommended)
2. Run all cells sequentially
3. Monitor training in TensorBoard (cell outputs)
4. If session disconnects: Re-run notebook, it will auto-resume
5. Final model saved to Google Drive in ONNX format

---

## Cell 1: Environment Setup

In [ ]:
# Check GPU and print environment info
!nvidia-smi

import sys
import torch

print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Verify mount
!ls /content/drive/MyDrive/

## Cell 3: Clone/Install Project Repository

In [ ]:
# Option A: Install from Drive (if you uploaded the project)
# %cd /content/drive/MyDrive/image-preprocessing-detector
# !pip install -q -e .[ml]

# Option B: Clone from GitHub (recommended)
!git clone https://github.com/YOUR_USERNAME/image-preprocessing-detector.git
%cd /content/image-preprocessing-detector

# Install dependencies
!pip install -q -e .
!pip install -q torch>=2.9.0 torchvision>=0.24.0 timm>=0.9.0 albumentations>=1.3.0
!pip install -q tensorboard wandb gdown

print("\n✅ Project installed!")

## Cell 4: Load Configuration & Setup Paths

In [ ]:
import yaml
from pathlib import Path

# Load training configuration
config_path = "/content/image-preprocessing-detector/configs/colab_phase2_iqa.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Setup paths in Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/image-preprocessing-detector"
DATASET_PATH = f"{DRIVE_ROOT}/datasets/iqa_phase2"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints/phase2_iqa"
LOGS_DIR = f"{DRIVE_ROOT}/logs/phase2_iqa"
MODEL_OUTPUT_DIR = f"{DRIVE_ROOT}/models/phase2_iqa"

# Create directories
for path in [CHECKPOINT_DIR, LOGS_DIR, MODEL_OUTPUT_DIR]:
    Path(path).mkdir(parents=True, exist_ok=True)

print("📁 Paths configured:")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Logs: {LOGS_DIR}")
print(f"   Models: {MODEL_OUTPUT_DIR}")

# Print configuration summary
print(f"\n⚙️  Training Configuration:")
print(f"   Model: {config['model']['architecture']}")
print(f"   Epochs: {config['training']['epochs']}")
print(f"   Batch Size: {config['training']['batch_size']}")
print(f"   Learning Rate: {config['training']['learning_rate']}")
print(f"   Mixed Precision: {config['training']['mixed_precision']['enabled']}")

## Cell 5: Initialize Colab Utilities & Check Session Health

In [ ]:
# Add project to path
sys.path.insert(0, '/content/image-preprocessing-detector')

from scripts.colab_utils import (
    print_environment_info,
    optimize_colab_environment,
    print_session_health,
    get_gpu_info
)

# Print full environment info
print_environment_info()

# Apply optimizations
optimize_colab_environment()

# Check session health
print_session_health()

# Get GPU tier for batch size adjustment
gpu_info = get_gpu_info()
gpu_tier = gpu_info.get('colab_tier', 'Unknown')
print(f"\n🎮 Detected GPU Tier: {gpu_tier}")

# Adjust batch size based on GPU
if 'T4' in gpu_tier:
    BATCH_SIZE = 32
    print("   Adjusted batch size to 32 for T4 GPU")
elif 'V100' in gpu_tier or 'P100' in gpu_tier:
    BATCH_SIZE = 64
    print("   Using batch size 64 for V100/P100 GPU")
elif 'A100' in gpu_tier:
    BATCH_SIZE = 128
    print("   Using batch size 128 for A100 GPU")
else:
    BATCH_SIZE = config['training']['batch_size']
    print(f"   Using default batch size {BATCH_SIZE}")

## Cell 6: Download/Prepare Dataset

**Note**: You need to have your dataset prepared in Google Drive.  
See `notebooks/colab/data_preparation.ipynb` for dataset generation.

In [ ]:
from scripts.gdrive_sync import download_dataset, check_drive_space

# Check available space
check_drive_space()

# Download dataset to local SSD for faster loading
LOCAL_DATASET_PATH = "/content/data"

if Path(DATASET_PATH).exists():
    print("\n📥 Downloading dataset from Google Drive to local SSD...")
    dataset_local = download_dataset(
        drive_path=DATASET_PATH,
        local_path=LOCAL_DATASET_PATH,
        extract_zip=True
    )
    print(f"✅ Dataset ready at: {dataset_local}")
else:
    print(f"\n⚠️  Dataset not found at {DATASET_PATH}")
    print("Please run data_preparation.ipynb first to generate the dataset.")
    print("Or upload your pre-prepared dataset to Google Drive.")
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

# Verify dataset structure
!ls -lh {LOCAL_DATASET_PATH}

## Cell 7: Create Data Loaders

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

# TODO: Replace with actual dataset class from your project
# This is a placeholder - you'll need to implement IQADataset
from image_preprocessing_detector.data.iqa_dataset import IQADataset  # Adjust import

# Define transforms
train_transform = A.Compose([
    A.Resize(config['model']['input_size'], config['model']['input_size']),
    A.RandomRotate90(p=0.5),
    A.Flip(p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(config['model']['input_size'], config['model']['input_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Create datasets
train_dataset = IQADataset(
    root=LOCAL_DATASET_PATH,
    split='train',
    transform=train_transform
)

val_dataset = IQADataset(
    root=LOCAL_DATASET_PATH,
    split='val',
    transform=val_transform
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=config['dataset']['num_workers'],
    pin_memory=config['dataset']['pin_memory'],
    prefetch_factor=config['dataset']['prefetch_factor']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=config['dataset']['num_workers'],
    pin_memory=config['dataset']['pin_memory']
)

print(f"📊 Dataset Statistics:")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Val samples: {len(val_dataset)}")
print(f"   Num classes: {config['model']['num_classes']}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## Cell 8: Create Model

In [ ]:
import timm
import torch.nn as nn

# Create model based on config
architecture = config['model']['architecture']
num_classes = config['model']['num_classes']
pretrained = config['model']['pretrained']

print(f"🏗️  Creating model: {architecture}")

# Load model from timm
model = timm.create_model(
    architecture,
    pretrained=pretrained,
    num_classes=num_classes
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Summary:")
print(f"   Architecture: {architecture}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {device}")
print(f"   Pretrained: {pretrained}")

## Cell 9: Setup Training (Optimizer, Loss, Scheduler)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Loss function (multi-label classification)
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)

# Learning rate scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode=config['training']['scheduler']['mode'],
    factor=config['training']['scheduler']['factor'],
    patience=config['training']['scheduler']['patience'],
    min_lr=config['training']['scheduler']['min_lr'],
    verbose=True
)

print("✅ Training components initialized:")
print(f"   Loss: BCEWithLogitsLoss (multi-label)")
print(f"   Optimizer: AdamW (lr={config['training']['learning_rate']})")
print(f"   Scheduler: ReduceLROnPlateau")

## Cell 10: Initialize Checkpoint Manager

In [ ]:
from scripts.checkpoint_manager import CheckpointManager

# Initialize checkpoint manager
checkpoint_manager = CheckpointManager(
    checkpoint_dir=CHECKPOINT_DIR,
    save_interval_epochs=config['checkpointing']['save_interval_epochs'],
    save_interval_minutes=config['checkpointing']['save_interval_minutes'],
    max_session_hours=config['platform']['auto_save_before_limit'],
    keep_last_n=config['checkpointing']['keep_last_n']
)

# Check if we should resume from checkpoint
start_epoch = 0
if checkpoint_manager.has_checkpoint():
    print("\n📂 Found existing checkpoint! Resuming training...")
    checkpoint_info = checkpoint_manager.load_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler
    )
    start_epoch = checkpoint_info['resume_epoch']
    print(f"✅ Resumed from epoch {start_epoch}")
else:
    print("\n🆕 No checkpoint found. Starting training from scratch.")

print(f"\n⏱️  Session monitoring:")
print(f"   Max session duration: {config['platform']['auto_save_before_limit']} hours")
print(f"   Checkpoint interval: Every {config['checkpointing']['save_interval_epochs']} epochs or {config['checkpointing']['save_interval_minutes']} minutes")

## Cell 11: Setup TensorBoard

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir=LOGS_DIR)

print(f"📈 TensorBoard initialized")
print(f"   Log directory: {LOGS_DIR}")

# Load TensorBoard in Colab
%load_ext tensorboard
%tensorboard --logdir {LOGS_DIR}

## Cell 12: Training Loop with Checkpoint Management

In [ ]:
from scripts.checkpoint_manager import train_with_checkpointing

# Start training with automatic checkpoint management
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"\nTotal epochs: {config['training']['epochs']}")
print(f"Starting from epoch: {start_epoch}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")
print(f"\nTraining will automatically stop at {config['platform']['auto_save_before_limit']} hours")
print("to save checkpoint before session limit.\n")

# Run training
results = train_with_checkpointing(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    checkpoint_manager=checkpoint_manager,
    num_epochs=config['training']['epochs'],
    device=device,
    scheduler=scheduler,
    start_epoch=start_epoch
)

print("\n" + "="*60)
print("✅ TRAINING COMPLETED!")
print("="*60)
print(f"\nFinal metrics: {results['final_metrics']}")

## Cell 13: Export Model to ONNX

In [ ]:
import torch.onnx

# Load best checkpoint
best_checkpoint_path = checkpoint_manager.get_best_checkpoint_path()
print(f"\n📦 Loading best model from: {best_checkpoint_path}")

checkpoint = torch.load(best_checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Export to ONNX
print("\n🔄 Exporting model to ONNX format...")

input_size = config['model']['input_size']
dummy_input = torch.randn(1, 3, input_size, input_size).to(device)

onnx_path = f"{MODEL_OUTPUT_DIR}/{architecture}_best.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=config['export']['onnx']['opset_version'],
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"✅ ONNX model saved to: {onnx_path}")

# Verify ONNX model
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✅ ONNX model verified successfully!")

# Print file size
import os
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"   Model size: {size_mb:.2f} MB")

## Cell 14: Final Summary & Next Steps

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 2 TRAINING COMPLETE!")
print("="*60)

print("\n📁 Output Locations:")
print(f"   Best Model (ONNX): {onnx_path}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   TensorBoard Logs: {LOGS_DIR}")

print("\n📊 Training Summary:")
print(f"   Total epochs trained: {len(results['history']['train_loss'])}")
print(f"   Best validation loss: {min(results['history']['val_loss']):.4f}")
print(f"   Best validation accuracy: {max(results['history']['val_accuracy']):.2f}%")

print("\n🔜 Next Steps:")
print("   1. Download ONNX model from Google Drive")
print("   2. Run model evaluation on test set (use model_evaluation.ipynb)")
print("   3. Integrate ONNX model into pipeline (src/detection/iqa_ml.py)")
print("   4. Test inference locally with ONNX Runtime")

print("\n✅ You can now close this session safely.")
print("   All artifacts are saved to Google Drive.")